# Paper 3 — 04 · Safety evaluation

Generates responses on the eval splits, judges them, computes safety
metrics (METHOD_DESIGN §5):

1. RoSafetyBench holdout (20% per dim).
2. Cross-lingual (eval-only, 86 prompts).
3. Held-out subcategory (Roma + Hungarian bias probes; train-disjoint).
4. EN HarmBench (regression check; English alignment must not break).
5. Moldovan-RO probe (~50 prompts; dialectal generalisation).

Reuses Paper 2's `gpt-5-mini` judge via `judges.py`. Captures `finish_reason`
+ token counts on every call (Paper 2 R10).

In [1]:
%%capture
# Pinned versions match METHOD_DESIGN §4 + requirements.txt. Restart-the-runtime
# is rarely needed because all of these are wheel-only on A100/CUDA 12.
!pip install -U \
    'transformers>=4.51' \
    'accelerate>=1.1' \
    'peft>=0.13' \
    'torchao>=0.16' \
    'trl>=0.11' \
    'datasets>=3.0' \
    'bitsandbytes>=0.44' \
    huggingface_hub ipywidgets pyyaml -q


In [2]:
import os, json, gc, time, hashlib, math, sys
from pathlib import Path
from datetime import datetime
import torch

# --- Drive ---
from google.colab import drive
drive.mount("/content/drive")

# --- Paths ---
DRIVE_ROOT  = Path("/content/drive/MyDrive/PhD/paper3-alignment")
PAPER2_ROOT = Path("/content/drive/MyDrive/PhD/paper2-benchmark")

PROBE_DIR    = DRIVE_ROOT / "data" / "probes"
PREFS_DIR    = DRIVE_ROOT / "data" / "preferences"
SPLITS_DIR   = DRIVE_ROOT / "data" / "splits"
ADAPTERS_DIR = DRIVE_ROOT / "adapters"
RESULTS_DIR  = DRIVE_ROOT / "results"
LOGS_DIR     = DRIVE_ROOT / "logs"
FIG_DIR      = DRIVE_ROOT / "figures"
for d in [PROBE_DIR, PREFS_DIR, SPLITS_DIR, ADAPTERS_DIR, RESULTS_DIR, LOGS_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Paper 2 reuse: judges, llm_judge, holdout splits ---
sys.path.insert(0, str(PAPER2_ROOT / "src"))

# --- A100 sanity + perf toggles ---
assert torch.cuda.is_available(), "GPU not available -- switch runtime to A100 GPU."
DEVICE_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {DEVICE_NAME}   VRAM: {VRAM_GB:.1f} GB   torch={torch.__version__}")
if "A100" not in DEVICE_NAME:
    print(f"WARNING: expected A100, got {DEVICE_NAME}. Re-tune BATCH_SIZE/grad-accum below.")

torch.set_float32_matmul_precision("high")          # TF32 for fp32 matmuls
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True               # autotune for fixed shapes

# --- HF / OpenRouter auth (set in Colab Secrets, not in code) ---
try:
    from google.colab import userdata
    if not os.environ.get("HF_TOKEN"):
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    if os.environ.get("HF_TOKEN"):
        from huggingface_hub import login as _hf_login
        _hf_login(os.environ["HF_TOKEN"], add_to_git_credential=False)
    if not os.environ.get("OPENROUTER_API_KEY"):
        os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY") or ""
except Exception as _e:
    print(f"(secrets not configured: {_e}; set HF_TOKEN / OPENROUTER_API_KEY in Colab secrets)")


def load_kwargs_for(family: str) -> dict:
    """A100-tuned dtype + attention impl per model family.

    Why bf16 everywhere: A100 has bf16 tensor cores at the same throughput as
    fp16, bf16 has the dynamic range of fp32 (no overflow on long sequences),
    and bf16 is the training dtype for all anchor families. Using fp16 here
    silently broke Gemma 3 in Paper 2 (953 empty-string outputs).
    """
    common = dict(torch_dtype=torch.bfloat16, device_map={"": 0})
    if family.startswith("gemma"):
        # Gemma 3 sliding-window attention is brittle with flash-attn-2.
        return {**common, "attn_implementation": "eager"}
    # PyTorch SDPA on A100 + bf16 is fast enough for our workloads.
    # flash-attn-2 was tried but its source-compile step costs ~15 min on
    # every cold Colab runtime, with negligible payoff for batch sizes <= 16.
    return {**common, "attn_implementation": "sdpa"}


def family_of(anchor_id: str) -> str:
    a = anchor_id.lower()
    if "qwen2.5" in a: return "qwen2.5"
    if "qwen3"   in a: return "qwen3"
    if "llama"   in a: return "llama"
    if "gemma"   in a: return "gemma"
    if "phi"     in a: return "phi"
    return "other"


def short_of(anchor_id: str) -> str:
    return (anchor_id.split("/")[-1].lower()
            .replace("-instruct", "").replace("-it", ""))

print("Bootstrap done.")


Mounted at /content/drive
GPU: NVIDIA A100-SXM4-80GB   VRAM: 85.1 GB   torch=2.10.0+cu128


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Bootstrap done.


## Configuration

In [3]:
ANCHOR    = "Qwen/Qwen2.5-3B-Instruct"
CONDITION = "rd-dpo-k4-bal-b03"
SEED      = 17

BATCH_GEN = 16
MAX_NEW   = 256

short  = short_of(ANCHOR)
family = family_of(ANCHOR)

RUN_TAG = f"{short}__{CONDITION}__seed{SEED}"
ADAPTER = ADAPTERS_DIR / RUN_TAG
OUT     = RESULTS_DIR / f"{RUN_TAG}__safety.json"
print(f"run    : {RUN_TAG}")
print(f"adapter: {ADAPTER if CONDITION != 'base' else '(none — base model)'}")

run    : qwen2.5-3b__rd-dpo-k4-bal-b03__seed17
adapter: /content/drive/MyDrive/PhD/paper3-alignment/adapters/qwen2.5-3b__rd-dpo-k4-bal-b03__seed17


## Load eval splits

In [4]:
SPLIT = json.loads((SPLITS_DIR / "split_v1.json").read_text())
by_id = {}
for dim_file in sorted((PAPER2_ROOT / "benchmark" / "expanded").glob("*.jsonl")):
    for line in open(dim_file, encoding="utf-8"):
        it = json.loads(line); by_id[(dim_file.stem, it["id"])] = it

EVAL_DIMS = ("toxicity", "jailbreak", "overrefusal", "bias",
             "hallucination", "crosslingual")
eval_holdout = {dim: [] for dim in EVAL_DIMS}
for entry in SPLIT["holdout"]:
    if entry["dim"] in EVAL_DIMS:
        eval_holdout[entry["dim"]].append(by_id[(entry["dim"], entry["id"])])
for dim, items in eval_holdout.items():
    print(f"{dim:>13s}  n={len(items)}")

     toxicity  n=53
    jailbreak  n=20
  overrefusal  n=20
         bias  n=43
hallucination  n=40
 crosslingual  n=86


## Load model (+ adapter)

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tok = AutoTokenizer.from_pretrained(ANCHOR, padding_side="left")
if tok.pad_token is None: tok.pad_token = tok.eos_token

base = AutoModelForCausalLM.from_pretrained(ANCHOR, **load_kwargs_for(family))
if CONDITION == "base":
    model = base
elif CONDITION == "dpo-full":
    model = AutoModelForCausalLM.from_pretrained(str(ADAPTER), **load_kwargs_for(family))
else:
    model = PeftModel.from_pretrained(base, str(ADAPTER))
model.eval()

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048)
        (layers): ModuleList(
          (0-30): 31 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
              (k_proj): Linear(in_features=2048, out_features=256, bias=True)
              (v_proj): Linear(in_features=2048, out_features=256, bias=True)
              (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
            )
            (mlp): Qwen2MLP(
              (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
              (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
              (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
              (act_fn): SiLUActivation()
            )
            (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
     

## Generate responses

Decoding params per METHOD §5.3: temp 1, top_p 0.95, max-new 256. We capture an
approximate `finish_reason` (saw-EOS vs hit-length).

In [6]:
@torch.inference_mode()
def generate_batch(prompts):
    formatted = [
        tok.apply_chat_template(
            [{"role": "user", "content": p}],
            tokenize=False, add_generation_prompt=True,
        ) for p in prompts
    ]
    responses, finish_reasons, gen_tokens = [], [], []
    for i in range(0, len(formatted), BATCH_GEN):
        batch = formatted[i:i + BATCH_GEN]
        inputs = tok(batch, return_tensors="pt", padding=True,
                     truncation=True, max_length=1024).to(model.device)
        out = model.generate(
            **inputs, max_new_tokens=MAX_NEW,
            do_sample=True, temperature=1.0, top_p=0.95,
            pad_token_id=tok.pad_token_id,
            return_dict_in_generate=True, output_scores=False,
        )
        input_len = inputs["input_ids"].shape[1]
        for seq in out.sequences:
            tail = seq[input_len:]
            resp = tok.decode(tail, skip_special_tokens=True)
            responses.append(resp[:1500])
            gen_tokens.append(int((tail != tok.pad_token_id).sum().item()))
            finish_reasons.append(
                "stop" if tok.eos_token_id in tail.tolist() else "length"
            )
    return responses, finish_reasons, gen_tokens

all_gen = {}
for dim, items in eval_holdout.items():
    if not items: continue
    prompts = [it["text_ro"] for it in items]
    t0 = time.time()
    resps, finish, ntok = generate_batch(prompts)
    all_gen[dim] = [
        {"id": items[j]["id"], "prompt": prompts[j], "response": resps[j],
         "finish_reason": finish[j], "completion_tokens": ntok[j]}
        for j in range(len(items))
    ]
    print(f"{dim:>13s}  {len(items)} prompts in {time.time()-t0:.1f}s")

     toxicity  53 prompts in 54.2s
    jailbreak  20 prompts in 26.0s
  overrefusal  20 prompts in 25.9s
         bias  43 prompts in 39.0s
hallucination  40 prompts in 38.8s
 crosslingual  86 prompts in 77.6s


## Score with Paper 2 judges

In [7]:
from llm_judge import Judge
from judges import REFUSAL_SYSTEM, REFUSAL_USER_TEMPLATE
from collections import Counter

judge = Judge(model="openai/gpt-5-mini", cache_namespace="paper3_eval")

judged = {}
for dim, items in all_gen.items():
    scored = []
    for it in items:
        v = judge.classify(
            system=REFUSAL_SYSTEM,
            user=REFUSAL_USER_TEMPLATE.format(
                prompt=it["prompt"], response=it["response"] or "[empty response]",
            ),
        )
        scored.append({**it, "judge_label": v.get("label"),
                       "judge_justification": v.get("justification", "")})
    judged[dim] = scored

from scipy.stats import binomtest
def wilson_ci(k, n, alpha=0.05):
    if n == 0: return (0.0, 0.0)
    return tuple(round(x, 3) for x in binomtest(k, n).proportion_ci(
        confidence_level=1 - alpha, method="wilson"))

safety = {}
for dim, items in judged.items():
    n = len(items)
    k = sum(1 for it in items if it["judge_label"] == "refusal")
    safety[dim] = {"refusal_rate": k / max(n, 1), "n": n,
                   "wilson_95": wilson_ci(k, n)}
print(json.dumps(safety, indent=2))

{
  "toxicity": {
    "refusal_rate": 0.4528301886792453,
    "n": 53,
    "wilson_95": [
      0.327,
      0.585
    ]
  },
  "jailbreak": {
    "refusal_rate": 0.6,
    "n": 20,
    "wilson_95": [
      0.387,
      0.781
    ]
  },
  "overrefusal": {
    "refusal_rate": 0.25,
    "n": 20,
    "wilson_95": [
      0.112,
      0.469
    ]
  },
  "bias": {
    "refusal_rate": 0.3488372093023256,
    "n": 43,
    "wilson_95": [
      0.224,
      0.498
    ]
  },
  "hallucination": {
    "refusal_rate": 0.2,
    "n": 40,
    "wilson_95": [
      0.105,
      0.348
    ]
  },
  "crosslingual": {
    "refusal_rate": 0.32558139534883723,
    "n": 86,
    "wilson_95": [
      0.236,
      0.43
    ]
  }
}


## Save

In [8]:
OUT.write_text(json.dumps({
    "anchor_model": ANCHOR, "condition": CONDITION, "seed": SEED,
    "adapter": str(ADAPTER) if CONDITION != "base" else None,
    "safety": safety, "judged": judged,
    "evaluated_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
}, ensure_ascii=False, indent=2))
print(f"wrote {OUT}")

wrote /content/drive/MyDrive/PhD/paper3-alignment/results/qwen2.5-3b__rd-dpo-k4-bal-b03__seed17__safety.json


/tmp/ipykernel_820/4040296433.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "evaluated_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",


## Batch eval: Llama + Gemma against the rebalanced beta=0.1 adapters

Loops over the two non-Qwen anchors. For each, loads the adapter
produced by notebook 03's batch cell, generates on the held-out
split, judges with gpt-5-mini, and writes
`results/<short>__rd-dpo-k4-bal__seed17__safety.json`.

Skips an anchor whose results file already exists, so safe to
re-execute.


In [ ]:
import gc, json, time
from datetime import datetime
from collections import Counter

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

from llm_judge import Judge
from judges import REFUSAL_SYSTEM, REFUSAL_USER_TEMPLATE
from scipy.stats import binomtest

BATCH_ANCHORS = [
    "meta-llama/Llama-3.2-3B-Instruct",
    "google/gemma-3-4b-it",
]
BATCH_CONDITION = "rd-dpo-k4-bal"
BATCH_SEED      = 17
BATCH_GEN       = 16
BATCH_MAX_NEW   = 256

def _wilson(k, n, alpha=0.05):
    if n == 0: return (0.0, 0.0)
    return tuple(round(x, 3) for x in binomtest(k, n).proportion_ci(
        confidence_level=1 - alpha, method="wilson"))


def _eval_one(anchor):
    short_a  = short_of(anchor)
    family_a = family_of(anchor)
    run_tag  = f"{short_a}__{BATCH_CONDITION}__seed{BATCH_SEED}"
    adapter_a = ADAPTERS_DIR / run_tag
    out_path  = RESULTS_DIR / f"{run_tag}__safety.json"

    if out_path.exists():
        print(f"[{anchor}] {out_path.name} already exists; skipping (delete to force rerun).")
        return
    if not adapter_a.exists():
        print(f"[{anchor}] adapter {adapter_a} not found; run notebook 03 batch cell first.")
        return

    print(f"\n=== evaluating {anchor} -> {out_path.name} ===")

    tok_a = AutoTokenizer.from_pretrained(anchor, padding_side="left")
    if tok_a.pad_token is None: tok_a.pad_token = tok_a.eos_token
    base_a = AutoModelForCausalLM.from_pretrained(anchor, **load_kwargs_for(family_a))
    model_a = PeftModel.from_pretrained(base_a, str(adapter_a))
    model_a.eval()

    @torch.inference_mode()
    def _generate_batch(prompts):
        formatted = [
            tok_a.apply_chat_template(
                [{"role": "user", "content": p}],
                tokenize=False, add_generation_prompt=True,
            ) for p in prompts
        ]
        responses, finish, ntok = [], [], []
        for i in range(0, len(formatted), BATCH_GEN):
            batch = formatted[i:i + BATCH_GEN]
            inputs = tok_a(batch, return_tensors="pt", padding=True,
                           truncation=True, max_length=1024).to(model_a.device)
            out = model_a.generate(
                **inputs, max_new_tokens=BATCH_MAX_NEW,
                do_sample=True, temperature=1.0, top_p=0.95,
                pad_token_id=tok_a.pad_token_id,
                return_dict_in_generate=True, output_scores=False,
            )
            input_len = inputs["input_ids"].shape[1]
            for seq in out.sequences:
                tail = seq[input_len:]
                resp = tok_a.decode(tail, skip_special_tokens=True)
                responses.append(resp[:1500])
                ntok.append(int((tail != tok_a.pad_token_id).sum().item()))
                finish.append("stop" if tok_a.eos_token_id in tail.tolist() else "length")
        return responses, finish, ntok

    all_gen_a = {}
    for dim, items in eval_holdout.items():
        if not items: continue
        prompts = [it["text_ro"] for it in items]
        t0 = time.time()
        resps, finish, ntok = _generate_batch(prompts)
        all_gen_a[dim] = [
            {"id": items[j]["id"], "prompt": prompts[j], "response": resps[j],
             "finish_reason": finish[j], "completion_tokens": ntok[j]}
            for j in range(len(items))
        ]
        print(f"  {dim:>13s}  {len(items)} prompts in {time.time()-t0:.1f}s")

    judge = Judge(model="openai/gpt-5-mini", cache_namespace="paper3_eval")
    judged_a = {}
    for dim, items in all_gen_a.items():
        scored = []
        for it in items:
            v = judge.classify(
                system=REFUSAL_SYSTEM,
                user=REFUSAL_USER_TEMPLATE.format(
                    prompt=it["prompt"], response=it["response"] or "[empty response]",
                ),
            )
            scored.append({**it, "judge_label": v.get("label"),
                           "judge_justification": v.get("justification", "")})
        judged_a[dim] = scored

    safety_a = {}
    for dim, items in judged_a.items():
        n = len(items)
        k = sum(1 for it in items if it["judge_label"] == "refusal")
        safety_a[dim] = {"refusal_rate": k / max(n, 1), "n": n,
                         "wilson_95": _wilson(k, n)}

    out_path.write_text(json.dumps({
        "anchor_model": anchor, "condition": BATCH_CONDITION, "seed": BATCH_SEED,
        "adapter": str(adapter_a),
        "safety": safety_a, "judged": judged_a,
        "evaluated_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    }, ensure_ascii=False, indent=2))
    print(f"  wrote {out_path}")
    print(f"  refusal: {[(d, round(safety_a[d]['refusal_rate']*100,1)) for d in safety_a]}")

    del model_a, base_a
    torch.cuda.empty_cache(); gc.collect()


for anchor in BATCH_ANCHORS:
    _eval_one(anchor)
print("\nbatch eval done.")
